In [ ]:
# 3.0 LLMs and Chat Models
from langchain.llms.openai import OpenAI
from langchain.chat_models import ChatOpenAI, ChatAnthropic

llm = OpenAI() # model = text-davinci-003
chat = ChatOpenAI() # model = gpt-3.5-turbo

a = llm.predict("How many planets are there?")
b = chat.predict("How many planets are there?")

a,b

In [ ]:
# 3.1 Predict Messages
from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage, AIMessage, SystemMessage

chat = ChatOpenAI(temperature=0.1)

messages = [
    SystemMessage(
        content="You are a geography expert. And you only reply in {language}.",
    ), # give context
    AIMessage(content="Ciao, mi chiamo {name}!"), # fake conversation
    HumanMessage(
        content="What is the distance between {country_a} and {country_b}. Also, what is your name?",
    ), # user question
]

chat.predict_messages(messages)

In [ ]:
#3.2 Prompt Templates: for predicting text

from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate # create template from string

chat = ChatOpenAI(temperature=0.1)

template = PromptTemplate.from_template("What is the distance between {country_a} and {country_b}")
prompt = template.format(country_a="Mexico", country_b="Thailand")

chat.predict(prompt)

In [ ]:
#3.2 Chat Prompt Templates: for predicting messages
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

chat = ChatOpenAI(temperature=0.1)

template = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert. And you only reply in {language}.")
    ("ai", "Ciao, mi chiamo {name}!"),
    ("human", "What is the distance between {country_a} and {country_b}. Also, what is your name?")
])

prompt = template.format_messages(
    language="Greek",
    name="Socrates",
    country_a="Mexico",
    country_b="Thailand",
)

chat.predict_messages(prompt)

In [ ]:
# 3.3 OutputParser: LLM Response 변형
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema import BaseOutputParser

class CommaOutputParser(BaseOutputParser):
    def parse(self, text):
        items = text.strip().split(",") # split into array
        return list(map(str.strip, items))
    
p = CommaOutputParser()
p.parse("Hello, how, are, you")

chat = ChatOpenAI(temperature=0.1)

template = ChatPromptTemplate.from_messages([
    ("system", "You are a list generating machine. Everything you are asked will be answered with a comma separated list of max {max_items} in lowercase.Do NOT reply with anything else."),
    ("human", "{question}"),
])

prompt = template.format_messages(
    max_items=10,
    question="What are the planets?"
)

result = chat.predict_messages(prompt)

p.parse(result.content)

In [ ]:
# 3.3 LCEL
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema import BaseOutputParser

class CommaOutputParser(BaseOutputParser):
    def parse(self, text):
        items = text.strip().split(",") # split into array
        return list(map(str.strip, items))

chat = ChatOpenAI(temperature=0.1)

template = ChatPromptTemplate.from_messages([
    ("system", "You are a list generating machine. Everything you are asked will be answered with a comma separated list of max {max_items} in lowercase.Do NOT reply with anything else."),
    ("human", "{question}"),
])

chain = template | chat | CommaOutputParser()
chain.invoke({
    "max_items": 5,
    "question": "What are the Pokemons",
})

In [ ]:
# 3.4 Chaining Chains
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

chef_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a world-class international chef. You create easy to follow recipes for any type of cuisine with easy to to find ingredients."),
    ("human", "I want to cook {cuisine} food."),
])

chef_chain = chef_prompt | llm

veg_chef_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a vegetarian chef specialized on making traditional recipies vegetarian. You find alternative ingredients and explain their preparation. You don't radically modify the recipe. If there is no alternative for a food just say you don't know how to replace it.",
        ),
        ("human", "{recipe}"),
    ]
)

veg_chain = veg_chef_prompt | chat

final_chain = {"recipe": chef_chain} | veg_chain
final_chain.invoke({
    "cuisine": "Indian"
})

In [ ]:
# 4.1 Without PromptTemplate.from_template
from langchain.prompts import PromptTemplate

t = PromptTemplate(
    template="What is the capital of {country}",
    input_variables=["country"],
)
t.format(country="France")

In [ ]:
# 4.1 FewShotPromptTemplate: 모델에게 대답 형식에 대해 예제를 준다.
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate, FewShotPromptTemplate

chat = ChatOpenAI()

examples = [
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

example_template = """
    Human: {question},
    AI: {answer}
"""

example_prompt_v1 = PromptTemplate.from_template(example_template)
example_prompt_v2 = PromptTemplate.from_template("Human: {question}\nAI: {answer}")

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt_v2,
    examples=examples,
    suffix="Human: What do you know about {country}?",
    input_variables=["country"]
)
prompt.format(country="Germany")

chain = prompt | chat
chain.invoke({
    "country": "Turkey"
})

In [ ]:
# 4.2 FewShotChatMessagePromptTemplate
from langchain.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

chat = ChatOpenAI()

examples = [
    {
        "country": "France",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "country": "Italy",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "country": "Greece",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "What do you know about {country}?"),
    ("ai", "{answer}")
])

example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert, you give short answers."),
    example_prompt,
    ("human", "What do you know about {country}?"),
])

chain = final_prompt | chat
chain.invoke({"country": "Thailand"})

In [ ]:
# 4.3 LengthBasedExampleSelector
from langchain.prompts import PromptTemplate, FewShotPromptTemplate
from langchain.prompts.example_selector import LengthBasedExampleSelector

examples = [
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

example_prompt = PromptTemplate.from_template("Human: {question}}\nAI: {answer}")

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=80,
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: What do you know about {country}?",
    input_variables=["country"],
)

chat = ChatOpenAI()

prompt.format(country="Brazil")

In [ ]:
# 4.3 Custom example selector: Random example selector
from langchain.prompts import PromptTemplate, FewShotPromptTemplate
from langchain.prompts.example_selector.base import BaseExampleSelector

examples = [
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

class RandomExampleSelector(BaseExampleSelector):

    def __init__(self, examples: list[dict[str, str]]):
        self.examples = examples

    def add_example(self, example: dict[str, str]):
        self.examples.append(example)

    def select_examples(self, input_variables):
        from random import choice
        return [choice(self.examples)]
    
example_prompt = PromptTemplate.from_template("Human: {question}\nAI: {answer}")

example_selector = RandomExampleSelector(
    examples=examples
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: What do you know about {country}?",
    input_variables=["country"],
)

prompt.format("Brazil")

In [ ]:
# 4.4 Serialization: Load prompt template from the disk
from langchain.prompts import load_prompt

json_prompt = load_prompt("./prompts/prompt.json")
yaml_prompt = load_prompt("./prompts/prompt.yaml")

yaml_prompt.format(country="German")

In [ ]:
#4.4 Composition: prompts 합치는 법
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.prompts.pipeline import PipelinePromptTemplate

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)

intro = PromptTemplate.from_template(
    """
    You are a role playing assistant.
    And you are impersonating a {character}
"""
)

example = PromptTemplate.from_template(
    """
    This is an example of how you talk:

    Human: {example_question}
    You: {example_answer}
"""
)

start = PromptTemplate.from_template(
    """
    Start now!

    Human: {question}
    You:
"""
)

final = PromptTemplate.from_template(
    """
    {intro}
                                     
    {example}
                              
    {start}
"""
)

prompts = [
    ("intro", intro),
    ("example", example),
    ("start", start),
]

full_prompt = PipelinePromptTemplate(
    final_prompt=final,
    pipeline_prompts=prompts,
)

chain = full_prompt | chat

chain.invoke(
    {
        "character": "Pirate",
        "example_question": "What is your location?",
        "example_answer": "Arrrrg! That is a secret!! Arg arg!!",
        "question": "What is your fav food?",
    }
)

In [ ]:
# 4.5 Caching: llm 답변을 메모리에 저장하여 재사용
from langchain.chat_models import ChatOpenAI
from langchain.globals import set_llm_cache
from langchain.cache import InMemoryCache

set_llm_cache(InMemoryCache())

chat = ChatOpenAI(temperature=0.1)
chat.predict("How do you make Italian Pasta?")

In [ ]:
# 4.5 Caching: llm 답변을 데이터베이스에 저장하여 재사용
from langchain.chat_models import ChatOpenAI
from langchain.globals import set_llm_cache
from langchain.cache import SQLiteCache

set_llm_cache(SQLiteCache("cache.db"))

chat = ChatOpenAI(temperature=0.1)
chat.predict("How do you make Italian Pasta?")

In [ ]:
# 4.5 Debug
from langchain.globals import set_debug

set_debug=True

chat = ChatOpenAI(temperature=0.1)
chat.predict("How do you make Italian Pasta?")

In [ ]:
#4.6 비용 계산
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import get_openai_callback

chat = ChatOpenAI(temperature=0.1)

with get_openai_callback as usage:
    a = chat.predict("What is the recipe for Soju?")
    b = chat.predict("How to make Bread")
    print(a,b,"\n")
    print(usage)
    print(usage.total_cost)

In [ ]:
#4.6 Serialization: 모델 저장 및 불러오기
from langchain.llms.openai import OpenAI
from langchain.llms.loading import load_llm

chat = OpenAI(
    temperature=0.1,
    max_tokens=450,
    model="gpt-3.5-turbo-16k"
)

chat.save("model.json")

loaded_chat = load_llm("model.json")

In [ ]:
# 5.0 ConversationBufferMemory: Saves whole conversation
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()
memory_for_chat_model = ConversationBufferMemory(return_messages=True)
memory.save_context({"input: Hi"}, {"output": "How are you"})
memory.load_memory_variables({})

In [ ]:
# 5.1 ConversationBufferWindowMemory: 최근 n개의 메시지만 저장
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(
    return_messages=True,
    k=2,
)

def add_message(input, output):
    memory.save_context({"input": input, "output": output})

add_message(1,1)
add_message(2,2)
add_message(3,3)

memory.load_memory_variables({})

In [ ]:
# 5.2 ConversationSummaryMemory
from langchain.memory import ConversationSummaryMemory
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(temperature=0.1)
memory = ConversationSummaryMemory(llm=llm)

def add_message(input, output):
    memory.save_context({"input": input}, {"output": output})

def get_history():
    memory.load_memory_variables({})

add_message("Hi, I'm Nico and I live in Seoul", "Wow, that is so cool")
add_message("South Korea is so pretty", "I wish i could go.")
get_history()

In [ ]:
# 5.3 ConversationSummaryBufferMemory: n개의 메시지부터 요약해서 저장한다.
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(temperature=0.1)
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=100,
    return_messages=True,
)

def add_message(input, output):
    memory.save_context({"input": input}, {"output": output})

def get_history():
    memory.load_memory_variables({})

add_message("Hi, I'm Nico and I live in Seoul", "Wow, that is so cool")
add_message("South Korea is so pretty", "I wish i could go.")
add_message("How far is Korea from Argentina?", "Dunno. Super far.")
get_history()

In [ ]:
# 5.4 ConversationKGMemory: 대화 중 중요한 내용만 뽑아내어 저장한다.
from langchain.memory import ConversationKGMemory
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(temperature=0.1)
memory = ConversationKGMemory(
    llm=llm,
    return_messages=True,
)

def add_message(input, output):
    memory.save_context({"input": input}, {"output": output})

add_message("Hi, I'm Nicolas and I live in Seoul", "Wow, that is so cool")
memory.load_memory_variables({"input": "Who is Nicolas?"})
add_message("Nicolas loves Kimchi", "Wow, that is so cool")
memory.load_memory_variables({"input": "What does Nicolas likes?"})

In [ ]:
# 5.5 Memory on LLMChain: off-the-shelf chain에 memory 설정
from langchain.chains import LLMChain
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=80,
    memory_key="chat_history",
)

template = """
    You are a helpful AI talking to human.

    {chat_history}
    Human: {question}
    You:
"""

chain = LLMChain(
    llm=llm,
    memory=memory,
    prompt=PromptTemplate.from_template(template),
    verbose=True,
)

chain.predict(question="My name is Nico")
chain.predict(question="I live in Seoul")
chain.predict(question="What is my name?")

memory.load_memory_variables({})

In [ ]:
# 5.6 Chat Based Memory
from langchain.chains import LLMChain
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=80,
    memory_key="chat_history",
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI talking to human."),
    MessagesPlaceholder(variable_name="chat_history"), # Getting unknown and unlimited messages
    ("human", "{question}"),
])

chain = LLMChain(
    llm=llm,
    memory=memory,
    prompt=prompt,
    verbose=True,
)

chain.predict(question="My name is Nico")
chain.predict(question="I live in Seoul")
chain.predict(question="What is my name?")

In [ ]:
# 5.7 LCEL Based Memory
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=80,
    memory_key="chat_history",
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI talking to human."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

chain = prompt | llm

chain.invoke({
    "chat_history": memory.load_memory_variables({})["chat_history"],
    "question": "My name is Nico"
})

In [ ]:
# 5.7 LCEL Based Memory_same result as above
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=80,
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI talking to human."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

def load_memory(_):
    return memory.load_memory_variables({})["history"]

chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm

def invoke_chain(question):
    result = chain.invoke({"question": question})
    memory.save_context({"input": question}, {"output": result.content})

In [ ]:
# 6.1 Data Loaders and Splitters
from langchain.document_loaders import TextLoader
from langchain.document_loaders import PyPDFLoader
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import CharacterTextSplitter

text_loader = TextLoader("./files/document.txt")
pdf_loader = PyPDFLoader("./files/document.pdf")
all_in_one_loader = UnstructuredFileLoader("./files/document.docx")

docs = text_loader.load()

# 문장이나 문단마다 끊어준다.
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200
) 
splitter.split_documents(docs)

separator_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=600, # max_characters
    chunk_overlap = 100,
)

all_in_one_loader.load_and_split(text_splitter=splitter)

In [ ]:
# 6.2 Tiktoken
from langchain.text_splitter import CharacterTextSplitter

splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=600,
    chunk_overlap = 100,
    length_function = len, # default
)

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    # count by token
    separator="\n",
    chunk_size=600,
    chunk_overlap = 100,
)

In [ ]:
# 6.4 Embedding
from langchain.embeddings import OpenAIEmbeddings

embedder = OpenAIEmbeddings()
vector = embedder.embed_query("Hi")
vector = embedder.embed_documents(["Hi", "How", "Are", "You"])

In [ ]:
# 6.4 Vector Store: Save Embeddings
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores.chroma import Chroma

loader = UnstructuredFileLoader("./files/document.txt")
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

docs = loader.load_and_split(text_splitter=splitter)
embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(docs, embeddings)

results = vectorstore.similarity_search("Where does the Winston live?")

In [ ]:
# 6.4 Cache Vector Store
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores.chroma import Chroma
from langchain.storage import LocalFileStore

cache_dir = LocalFileStore("./.cache/")

loader = UnstructuredFileLoader("./files/document.txt")
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = Chroma.from_documents(docs, cached_embeddings)

In [ ]:
# 6.6 RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores.chroma import Chroma
from langchain.storage import LocalFileStore
from langchain.chains import RetrievalQA

cache_dir = LocalFileStore("./.cache/")

llm = ChatOpenAI()
loader = UnstructuredFileLoader("./files/document.txt")
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = Chroma.from_documents(docs, cached_embeddings)

chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(),
)

chain.run("Describe Victory Mansions")

In [ ]:
# 6.8 Stuff LCEL Chain
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores.chroma import Chroma
from langchain.storage import LocalFileStore
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough

cache_dir = LocalFileStore("./.cache/")

llm = ChatOpenAI(temperature=0.1)
loader = UnstructuredFileLoader("./files/document.txt")
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = Chroma.from_documents(docs, cached_embeddings)

retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer questions using only following context. If you don't know the answer, just say you don't know. Don't make it up:\n\n{context}"),
    ("human", "{question}"),
])

chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm
chain.invoke("Describe Victory Mansions")


In [ ]:
# 6.9 Map Reduce LCEL Chain
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores.chroma import Chroma
from langchain.storage import LocalFileStore
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda

cache_dir = LocalFileStore("./.cache/")

llm = ChatOpenAI(temperature=0.1)
loader = UnstructuredFileLoader("./files/document.txt")
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = Chroma.from_documents(docs, cached_embeddings)

retriever = vectorstore.as_retriever()

map_doc_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Use the following portion of a long document to see if any of the text is relevant to answer the question. Return any relevant text verbatim. If there is no relevant text, return : ''
            -------
            {context}
            """,
        ),
        ("human", "{question}"),
    ]
)

map_doc_chain = map_doc_prompt | llm


def map_docs(inputs):
    documents = inputs["documents"]
    question = inputs["question"]
    return "\n\n".join(
        map_doc_chain.invoke({
            "context": document.page_content,
            "question": question
        }).content for document in documents
    )

map_chain = {
    "documents": retriever,
    "question": RunnablePassthrough(),
} | RunnableLambda(map_docs)

final_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Given the following extracted parts of a long document and a question, create a final answer. 
            If you don't know the answer, just say that you don't know. Don't try to make up an answer.
            ------
            {context}
            """,
        )("human", "{question}")
    ]
)

chain = {"context": map_chain, "question": RunnablePassthrough()} | final_prompt | llm

chain.invoke("Where does Winston goes to work?")

In [ ]:
# 8.1 HuggingFaceHub: HuggingFace server
from langchain.llms.huggingface_hub import HuggingFaceHub
from langchain.prompts import PromptTemplate

prompt = PromptTemplate.from_template("[INST]What is the meaning of {word}[/INST]")
llm = HuggingFaceHub(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2", 
    model_kwargs={"max_new_tokens": 1000},
)

chain = prompt | llm
chain.invoke({"word": "potato"})

In [ ]:
# 8.2 HuggingFacePipeline: Download model
from langchain.llms.huggingface_pipeline import HuggingFacePipeline

prompt = PromptTemplate.from_template("A {word} is a")

llm = HuggingFacePipeline.from_model_id(
    model_id="gpt2",
    task="text-generation",
    pipeline_kwargs={"max_new_tokens": 50},
)

chain = prompt | llm

chain.invoke({"word": "tomato"})

In [ ]:
# 8.3 GPT4All: Download model
from langchain.llms.gpt4all import GPT4All

llm = GPT4All(
    model="./falcon.bon", # where to save model,
    device="gpu",
) 

prompt = PromptTemplate.from_template("You are a helpful assistant that defines words. Define this word: {word}.")

chain = prompt | llm

chain.invoke({"word": "potato"})

In [ ]:
# 9.8 Function Calling
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate

def get_weather(lon, lat):
    return

function = {
    "name": "get_weather",
    "description": "function that takes longitude and latitude to find the weather of the place",
    "parameters": {
        "type": "object",
        "properties": {
            "lon": {"type":float, "description":"The longitude coordinate"},
            "lat": {"type":float, "description":"The latitude coordinate"},
        }
    },
    "required": ["lon", "lat"],
}

llm = ChatOpenAI(temperature=0.1).bind(
    function_call={"name": "get_weather"}, # force the model to use the function
    function_call="auto", # 모델이 자율적으로 선택
    functions=[function]
)
prompt = PromptTemplate.from_template("How's the weather in {city}")
chain = prompt | llm
response = chain.invoke({"city":"rome"})
'''reponse
AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{\n "lon": "12.4964",\n "lat": "41.9208"\n}'}})
'''
response = response.addtional_kwargs["function_call"]["arguments"]

import json
r = json.loads(response)
get_weather(r['lon'], r['lat'])

In [ ]:
# 9.8 Function Calling
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate


function = {
    "name": "create_quiz",
    "description": "function that takes a list of questions and answers and returns a quiz",
    "parameters": {
        "type": "object",
        "properties": {
            "questions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "question": {
                            "type": "string",
                        },
                        "answers": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "answer": {
                                        "type": "string",
                                    },
                                    "correct": {
                                        "type": "boolean",
                                    },
                                },
                                "required": ["answer", "correct"],
                            },
                        },
                    },
                    "required": ["question", "answers"],
                },
            }
        },
        "required": ["questions"],
    },
}


llm = ChatOpenAI(
    temperature=0.1,
).bind(
    function_call={
        "name": "create_quiz",
    },
    functions=[
        function,
    ],
)

prompt = PromptTemplate.from_template("Make a quiz about {city}")

chain = prompt | llm

response = chain.invoke({"city": "rome"})


response = response.additional_kwargs["function_call"]["arguments"]

import json
for question in json.loads(response)['questions']:
    print(question)